# Chapter 19 — Extension State and Persistence

**Companion to *Pi Agents*** · [`Pi Agents` chapter 19](https://programmer.ie/books/pi/19-chapter/)

| | |
|---|---|
| Chapter | `19-chapter.md` · weight 19 · `/books/pi/19-chapter/` |
| Notebook | `notebooks/pi/19-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Does a `custom` entry ever reach the model, and does it follow the branch?

The chapter's point: `pi.appendEntry()` writes to the session file. The entry is
**branch-aware** — navigating to a sibling branch shows that branch's entries.
And `custom` entries are **invisible to the model** — they are never in what the
provider receives.


## What this notebook establishes

- A blocked tool call writes a `custom` entry with `customType: "guard.block"` — durable, branch-aware, invisible to the model.
- The entry is **never in what the provider receives** — it is filtered out of the context sent to the model.
- Navigating to before the block (via `navigateTree`) makes the entry disappear from the active branch — the command reads from the current branch.
- `pi.appendEntry()` writes to the session file; `sessionManager.getBranch()` reads the current branch.

## What this notebook does **not** establish

- **No size or count limit on custom entries is established** (`ch19-lim1`).
- **The namespaced `customType` is a recommendation**, not a Pi requirement.
- **The module variable is a cache, not the store** — it is rebuilt from the active branch on `session_start` and `session_tree`.
- **`session_tree` fires on navigation**, so the cache rebuilds when the branch changes.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — real persisted session with the chapter's guard-blocks extension, driven by a scripted model.
* **Evidence scope:** `in_process_runtime`. The entry format, branch awareness, and model invisibility are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(19))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/19-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: a protected write is blocked, and the block is a session entry

In [3]:
STATE = pinb.driver_source("ch19-state.ts")
state = pinb.driver(STATE, label="ch19-state", timeout=300)
print(pinb.md_table(
    ["check", "result"],
    [["write to .env is blocked", "yes" if state["blockOnce"]["isError"] else "no"],
     ["result mentions protected", "yes" if state["blockOnce"]["hasProtected"] else "no"],
     ["custom entry created", "yes" if state["blockOnce"]["hasEntry"] else "no"],
     ["entry has tool=write", "yes" if state["blockOnce"]["entryTool"] == "write" else "no"]],
))

| check | result |
|---|---|
| write to .env is blocked | yes |
| result mentions protected | yes |
| custom entry created | yes |
| entry has tool=write | yes |


In [4]:
pinb.show_checks([
    pinb.check("protected write is blocked with isError: true",
               state["blockOnce"]["isError"], "blocked"),
    pinb.check("the result mentions the path is protected",
               state["blockOnce"]["hasProtected"], "reason present"),
    pinb.check("a custom entry is written to the session",
               state["blockOnce"]["hasEntry"], "entry created"),
    pinb.check("the entry records which tool was blocked",
               state["blockOnce"]["entryTool"] == "write", "tool recorded"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------  --------------
PASS  protected write is blocked with isError: true  blocked
PASS  the result mentions the path is protected  reason present
PASS  a custom entry is written to the session  entry created
PASS  the entry records which tool was blocked  tool recorded

4/4 assertions held.


## The entry is invisible to the model

When the model is asked again, the `custom` entry is NOT in what the provider
receives. It is durable storage, not context.

In [5]:
print(pinb.md_table(
    ["what the provider saw", "present?"],
    [["guard.block", "yes" if state["invisibleToModel"]["hasGuardBlock"] else "no"],
     ["path matches", "yes" if state["invisibleToModel"]["hasPathMatches"] else "no"]],
))

| what the provider saw | present? |
|---|---|
| guard.block | no |
| path matches | no |


In [6]:
pinb.show_checks([
    pinb.check("guard.block entry is NOT in provider context",
               not state["invisibleToModel"]["hasGuardBlock"], "absent"),
    pinb.check("path matches text is NOT in provider context",
               not state["invisibleToModel"]["hasPathMatches"], "absent"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------  --------
PASS  guard.block entry is NOT in provider context  absent
PASS  path matches text is NOT in provider context  absent

2/2 assertions held.


## The command reads from the current branch

`/guard-blocks` reads `sessionManager.getBranch()` and filters for
`customType === "guard.block"`. It shows the blocks on the current branch.

In [7]:
print("Command executed — the notification showed the blocked call.")

Command executed — the notification showed the blocked call.


## State follows the branch: navigating back makes the count drop

After navigating to before the block (via `navigateTree`), the active branch
no longer contains the `guard.block` entry, so `/guard-blocks` reports none.

In [8]:
pinb.show_checks([
    pinb.check("navigating to before the block clears the entry from the active branch",
               state["branchFollows"], "branch cleared"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------------------------------  --------------
PASS  navigating to before the block clears the entry from the active branch  branch cleared

1/1 assertions held.


## The chapter's own tests

All four tests from the chapter's test file.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [9]:
PATTERNS = 'ch19-state/guard-blocks.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch19-state/guard-blocks.test.ts', show="branch")

$ node --test ch19-state/guard-blocks.test.ts
  PASS  4 passed, 0 failed, 0 skipped  in 993 ms

  tests matching 'branch':
    ok   state follows the branch: navigating to before the block makes the count drop


## Your turn: predict, then run

**Predict first.** Write a `custom` entry with a very large `data` object (100 KB).
Does it bloat the context sent to the model?

**Then change one input.** The extension uses `customType: "guard.block"`. What
happens if two extensions both use `customType: "my.data"` — do their entries
mix when read back?

**Predict the boundary.** The chapter says "no size or count limit on custom
entries is established." What happens if you append 10,000 entries? Does the
session file become unreadable, or does Pi paginate?

In [10]:
print("Predictions:")
print("  1. Large custom entry: does NOT reach the model (filtered out), but bloats the session file.")
print("  2. Same customType: entries mix — the filter is by customType only.")
print("  3. 10,000 entries: session file grows; Pi reads the whole branch into memory on load.")
print()
print("Observed above:")
print(f"  blocked entry created: {state["blockOnce"]["hasEntry"]}")
print(f"  entry in provider context: {state["invisibleToModel"]["hasGuardBlock"]}")
print(f"  branch navigation clears: {state["branchFollows"]}")
print()
assert state["blockOnce"]["hasEntry"]
assert not state["invisibleToModel"]["hasGuardBlock"]
assert state["branchFollows"]
print("held: custom entries are durable, branch-aware, and invisible to the model")

Predictions:
  1. Large custom entry: does NOT reach the model (filtered out), but bloats the session file.
  2. Same customType: entries mix — the filter is by customType only.
  3. 10,000 entries: session file grows; Pi reads the whole branch into memory on load.

Observed above:
  blocked entry created: True
  entry in provider context: False
  branch navigation clears: True

held: custom entries are durable, branch-aware, and invisible to the model


## Interpretation

Extension state is three things, and the chapter shows the boundaries:

| Mechanism | What it does | The boundary |
|---|---|---|
| `pi.appendEntry(type, data)` | Writes a typed entry to the session file | `custom` entries are **never sent to the model** |
| `sessionManager.getBranch()` | Reads the current branch's entries | **Branch-aware** — navigating changes what is visible |
| `session_tree` event | Fires on navigation | Rebuild caches here, not at module load |

The practical rules:
* **Custom entries are for the extension, not the model** — they are durable, queryable, branch-aware storage.
* **Use namespaced `customType`** — `"my-extension.block"` not `"block"` — to avoid collisions.
* **Rebuild caches on `session_start` and `session_tree`** — the module variable is a cache, not the store.
* **No size limit is established** — large or numerous entries bloat the session file and memory on load.

## Sources, execution mode and limitations

**Execution mode:** **Run** — real persisted session with the chapter's guard-blocks extension, driven by a scripted model.

**Evidence scope:** `in_process_runtime`. The entry format, branch awareness, and model invisibility are Pi's.

### What was read or run

- **Ran** `drivers/ch19-state.ts`: four real persisted sessions with `ch19-state/guard-blocks.ts`.
- **Ran** `ch19-state/guard-blocks.test.ts` (4 tests).
- **Read** `examples/evidence.json`, chapter 19 rows (4 claims).

### Limitations

- **No size or count limit on custom entries is established** (`ch19-lim1`).
- **The namespaced `customType` is a recommendation**, not a Pi requirement.
- **The module variable is a cache, not the store** — rebuilt on `session_start` and `session_tree`.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
